# Advanced Machine Learning — D2.2

**Forecasting daily IFR flight movements at Palma de Mallorca Airport (LEPA)**

*Xabier Villa López & Illart Dekhli Moreno*

This notebook builds on our D2.1 plan: we are forecasting daily IFR flights at LEPA 14 days ahead. We covered the dataset choice, stationarity argument and prior work in D2.1, so here we go straight into the modelling work required by D2.2.

The required sections are: time series analysis, ARMA, ARIMA, SARIMA, AutoArima, a Transformer-based model, results discussion and lessons learned. Before any of that we set up an evaluation harness and a few baselines, so that every model is compared on the same metric and the same test windows.

## 0. Setup

In [ ]:
# Install the libraries that are not preinstalled in Colab.
# Uncomment if running on a fresh Colab kernel.
# !pip install -q pmdarima darts

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Data Loading

The data comes from the Eurocontrol Aviation Intelligence Portal (https://www.ansperformance.eu/data/). They publish a daily IFR traffic file with one row per airport per day. We filter it to `LEPA` (Palma de Mallorca) and use the total IFR movements column (arrivals + departures) as our target.

To run this on Colab, upload the Eurocontrol CSV to a Drive folder and set `DATA_PATH` below to the file location (or the public download URL). Locally, just point it at the file on disk.

In [ ]:
# Path to the Eurocontrol CSV. Adjust to wherever you have it.
DATA_PATH = "Airport_Traffic.csv"

# Column names in the Eurocontrol file. If the file you downloaded uses different
# names, print df_raw.columns below and update these constants.
DATE_COL = "FLT_DATE"
ICAO_COL = "APT_ICAO"
TOTAL_IFR_COL = "FLT_TOT_IFR_2"   # total daily IFR movements (dep + arr)

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(df_raw):,}")
print("Columns:", list(df_raw.columns))
df_raw.head()

In [ ]:
# Filter to LEPA and keep only what we need
df = (
    df_raw[df_raw[ICAO_COL] == "LEPA"]
    .loc[:, [DATE_COL, TOTAL_IFR_COL]]
    .copy()
)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.set_index(DATE_COL).sort_index()
df = df.rename(columns={TOTAL_IFR_COL: "flights"})

# Reindex to a continuous daily frequency to expose any gaps
full_index = pd.date_range(df.index.min(), df.index.max(), freq="D")
df = df.reindex(full_index)
df.index.name = "date"

print(f"Date range: {df.index.min().date()} -> {df.index.max().date()}")
print(f"Total days:   {len(df):,}")
print(f"Missing days: {int(df['flights'].isna().sum())}")
df.head()

In [ ]:
# If there are isolated gaps (rare), fill them using the same weekday a week before/after
if df["flights"].isna().any():
    df["flights"] = df["flights"].fillna(df["flights"].shift(7))
    df["flights"] = df["flights"].fillna(df["flights"].shift(-7))
    print(f"After filling, missing: {int(df['flights'].isna().sum())}")

series_full = df["flights"].astype(float)

### 1.1 Handling the COVID structural break

The COVID drop in 2020 is so large that it will dominate any model trained on the full series, and the recovery in 2020-2021 is not representative of normal operations either. To compare model families fairly (from ARMA all the way to a Transformer), we cut the series at **2022-01-01** and treat that as our analysis period. We mention this trade-off in the lessons learned section.

In [ ]:
COVID_CUT = "2022-01-01"

series = series_full.loc[COVID_CUT:]
print(f"Modern series: {series.index.min().date()} -> {series.index.max().date()}")
print(f"Length:        {len(series):,} days  (~{len(series)/365:.1f} years)")

## 2. Section 1 — Time Series Analysis

We visualise the series, look at the two seasonal cycles we expect (weekly and yearly), run STL decomposition, and apply the stationarity tests we promised in D2.1.

In [ ]:
# Full series — for the record, with the excluded COVID period highlighted
fig, ax = plt.subplots(figsize=(13, 4))
series_full.plot(ax=ax, color="steelblue", linewidth=0.8)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp(COVID_CUT),
           color="red", alpha=0.12, label="COVID period (excluded)")
ax.set_title("Daily IFR movements at LEPA — full Eurocontrol series")
ax.set_ylabel("Flights / day")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Modern series — what we will actually model
fig, ax = plt.subplots(figsize=(13, 4))
series.plot(ax=ax, color="steelblue", linewidth=0.8)
ax.set_title(f"LEPA daily IFR — analysis series ({series.index.min().date()} onwards)")
ax.set_ylabel("Flights / day")
plt.tight_layout()
plt.show()

In [ ]:
# Weekly seasonality — boxplot by day of week
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df_dow = series.to_frame("flights").assign(dow=series.index.dayofweek)

fig, ax = plt.subplots(figsize=(8, 4))
df_dow.boxplot(column="flights", by="dow", ax=ax)
ax.set_xticklabels(DAY_NAMES)
ax.set_title("Flights by day of week")
ax.set_xlabel("")
ax.set_ylabel("Flights / day")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Yearly seasonality — boxplot by month of year
df_m = series.to_frame("flights").assign(month=series.index.month)
fig, ax = plt.subplots(figsize=(10, 4))
df_m.boxplot(column="flights", by="month", ax=ax)
ax.set_title("Flights by month of year")
ax.set_xlabel("Month")
ax.set_ylabel("Flights / day")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# STL decomposition with weekly seasonality (the strongest short cycle in daily data)
stl = STL(series, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

### 2.1 Stationarity tests

We promised in D2.1 to run both ADF (null = unit root, i.e. random-walk-like) and KPSS (null = stationary). Together they tell us whether the series, or a differenced version of it, looks stationary enough for the ARMA family.

In [ ]:
def report_adf(x, name=""):
    stat, p, *_ = adfuller(x.dropna(), autolag="AIC")
    verdict = "reject unit root (stationary-ish)" if p < 0.05 else "cannot reject unit root"
    print(f"ADF  on {name:20s}  stat={stat:7.3f}  p={p:.4f}   -> {verdict}")

def report_kpss(x, name=""):
    stat, p, *_ = kpss(x.dropna(), regression="c", nlags="auto")
    verdict = "reject stationarity" if p < 0.05 else "cannot reject stationarity"
    print(f"KPSS on {name:20s}  stat={stat:7.3f}  p={p:.4f}   -> {verdict}")

print("Raw series")
report_adf(series, "raw")
report_kpss(series, "raw")
print()

# Seasonally differenced (lag 7) to remove the weekly cycle
series_d7 = series.diff(7)
print("Lag-7 differenced")
report_adf(series_d7, "diff_7")
report_kpss(series_d7, "diff_7")

In [ ]:
# ACF / PACF after seasonal differencing — useful for picking SARIMA orders later
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(series_d7.dropna(), lags=40, ax=axes[0])
plot_pacf(series_d7.dropna(), lags=40, ax=axes[1], method="ywm")
axes[0].set_title("ACF — lag-7 differenced")
axes[1].set_title("PACF — lag-7 differenced")
plt.tight_layout()
plt.show()

## 3. Evaluation Harness

Every model in this notebook is evaluated the same way: rolling 14-day forecasts over the last few months of the series, with **MAE in flights/day** as the comparison metric. We chose MAE because the numbers are directly interpretable as "average error in flights per day", which makes the discussion section a lot easier than with sMAPE or RMSE.

We hold out the last `HORIZON * N_WINDOWS` days and step through them in non-overlapping chunks. Models that need to be refit at every step (ARMA / ARIMA / SARIMA) will be refit; baselines just slide forward.

In [ ]:
HORIZON = 14
N_WINDOWS = 12       # roughly 6 months of test
STEP = HORIZON       # non-overlapping windows

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred))**2)))

def get_windows(series, horizon=HORIZON, n_windows=N_WINDOWS, step=STEP):
    '''Yield (train, actual) pairs for rolling-origin evaluation.'''
    test_size = n_windows * step
    train_end = len(series) - test_size
    assert train_end > 365, "Not enough history before the test windows"
    for i in range(n_windows):
        cutoff = train_end + i * step
        train = series.iloc[:cutoff]
        actual = series.iloc[cutoff:cutoff + horizon]
        yield train, actual

def evaluate(forecast_fn, series=series, horizon=HORIZON,
             n_windows=N_WINDOWS, step=STEP, name=""):
    '''
    forecast_fn(train_series, horizon) -> 1D array of `horizon` predictions.
    Returns a dict with mean MAE/RMSE plus per-window errors.
    '''
    maes, rmses = [], []
    for train, actual in get_windows(series, horizon, n_windows, step):
        preds = np.asarray(forecast_fn(train, horizon))
        assert len(preds) == horizon, f"forecast_fn returned {len(preds)} values, expected {horizon}"
        maes.append(mae(actual.values, preds))
        rmses.append(rmse(actual.values, preds))
    return {"model": name,
            "mae": float(np.mean(maes)),
            "rmse": float(np.mean(rmses)),
            "per_window_mae": maes}

# Sanity check — see what our test windows actually cover
windows = list(get_windows(series))
print(f"Number of test windows: {len(windows)}")
print(f"First window: train ends {windows[0][0].index[-1].date()}, "
      f"forecast {windows[0][1].index[0].date()} -> {windows[0][1].index[-1].date()}")
print(f"Last window:  train ends {windows[-1][0].index[-1].date()}, "
      f"forecast {windows[-1][1].index[0].date()} -> {windows[-1][1].index[-1].date()}")

In [ ]:
# Master results table — every model appends one row
results = []
def add_result(r):
    results.append(r)
    return r

## 4. Baselines

Three sanity-check baselines we will keep on the leaderboard:

1. **Last value** — predict every future day as the most recent observed value
2. **Mean** — predict every future day as the historical mean of the training portion
3. **Seasonal naive (lag 7)** — repeat the last week

Any real model needs to beat seasonal naive lag 7 to justify its existence on this series.

In [ ]:
def baseline_last(train, h):
    return np.repeat(train.iloc[-1], h)

def baseline_mean(train, h):
    return np.repeat(train.mean(), h)

def baseline_seasonal_naive_7(train, h):
    last_week = train.iloc[-7:].values
    reps = int(np.ceil(h / 7))
    return np.tile(last_week, reps)[:h]

for name, fn in [
    ("Baseline — last value",            baseline_last),
    ("Baseline — mean",                  baseline_mean),
    ("Baseline — seasonal naive (7)",    baseline_seasonal_naive_7),
]:
    r = evaluate(fn, name=name)
    add_result(r)
    print(f"{r['model']:35s}  MAE={r['mae']:7.2f}   RMSE={r['rmse']:7.2f}")

In [ ]:
# Quick visual check on the first test window — actual vs each baseline
train, actual = windows[0]
fig, ax = plt.subplots(figsize=(11, 4))
actual.plot(ax=ax, label="Actual", color="black", linewidth=2)
ax.plot(actual.index, baseline_last(train, HORIZON),              label="Last value",        linestyle="--")
ax.plot(actual.index, baseline_mean(train, HORIZON),              label="Mean",              linestyle="--")
ax.plot(actual.index, baseline_seasonal_naive_7(train, HORIZON),  label="Seasonal naive (7)", linestyle="--")
ax.set_title(f"Baselines on first test window  ({actual.index[0].date()} -> {actual.index[-1].date()})")
ax.set_ylabel("Flights / day")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Section 2 — ARMA

*To do.* ARMA assumes a stationary series with no integrated part. The KPSS test on the raw series will probably reject stationarity because of the yearly cycle, so plain ARMA is not really applicable on the raw values. We have two options:

- **Justify skipping it** — the rubric explicitly allows this if we explain why
- **Fit ARMA on the seasonally-differenced (lag 7) series** — as a curiosity to compare with ARIMA

Decide with Xabi before writing this up.

## 6. Section 3 — ARIMA

*To do.* Fit two `(p, d, q)` combinations on the raw series, with `d` picked from the ADF result above. The rubric also asks for a "naïve seasonal approach using ARIMA" — that probably means doing a lag-7 seasonal difference manually before fitting ARIMA, so the model only has to learn the deseasonalised dynamics.

## 7. Section 4 — SARIMA

*To do.* SARIMA with weekly seasonality `m=7`. We pick orders from the ACF/PACF plots above. Full `m=365` SARIMA is computationally not viable, so the yearly cycle will be absorbed either by the model's drift term, by a small number of yearly Fourier terms as exogenous regressors, or simply by having short enough forecast horizons that the yearly cycle barely moves within them.

## 8. Section 5 — AutoArima

*To do.* Run `pmdarima.auto_arima` twice: once with `seasonal=False`, once with `seasonal=True, m=7`. The non-seasonal version is mostly there because the rubric explicitly asks for it; we expect the seasonal one to win.

## 9. Section 6 — Transformer-based model

*To do.* Use Darts' `TransformerModel` (or `TFTModel` if we want to be fancier) on the same train/test splits. Lookback window around 60-90 days, output horizon = 14. Remember to scale inputs and set torch seeds for reproducibility. Same MAE harness.

## 10. Section 7 — Results discussion

*To do.* Final results table, observations, what surprised us. Use the `results` list we have been appending to and turn it into a pandas DataFrame for the comparison table.

## 11. Section 8 — Lessons Learned

*To do.* Challenges, decisions (COVID cut, `m=7` instead of `m=365`, MAE choice), insights on window / horizon / parameter selection.